# Bundesliga — App de Predicciones

**Requisito:** `modelo_german.pkl` generado por `german_league_train.ipynb`

```
Antes de la jornada     →  Celdas 1, 2, 3, 4, 5
Después de cada partido →  Celda 6
Ver resumen             →  Celda 7
```

**Umbral de uso:** Solo usar predicciones con confianza > 55% (marcadas `← USAR`)

In [3]:
# ─────────────────────────────────────────────
# CELDA 1 — Imports y carga del modelo
# Correr siempre al abrir el notebook
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import joblib
import json

datos = joblib.load('modelo_german.pkl')

model_v3        = datos['model_v3']
feature_cols_v3 = datos['feature_cols_v3']
le              = datos['le']
df              = datos['df']

print('Modelo cargado correctamente')
print(f'  Fecha entrenamiento:   {datos["fecha_entrenamiento"]}')
print(f'  Accuracy:              {datos["accuracy"]:.1%}')
print(f'  Accuracy alta conf:    {datos["accuracy_alta_confianza"]:.1%}')
print(f'  Features:              {len(feature_cols_v3)}')
print(f'\nEquipos disponibles:')
print(sorted(df['HomeTeam'].unique()))

Modelo cargado correctamente
  Fecha entrenamiento:   2026-04-08
  Accuracy:              53.8%
  Accuracy alta conf:    64.4%
  Features:              52

Equipos disponibles:
['Augsburg', 'Bayern Munich', 'Bielefeld', 'Bochum', 'Darmstadt', 'Dortmund', 'Ein Frankfurt', 'FC Koln', 'Freiburg', 'Greuther Furth', 'Hamburg', 'Heidenheim', 'Hertha', 'Hoffenheim', 'Holstein Kiel', 'Leverkusen', "M'gladbach", 'Mainz', 'RB Leipzig', 'Schalke 04', 'St Pauli', 'Stuttgart', 'Union Berlin', 'Werder Bremen', 'Wolfsburg']


In [4]:
# ─────────────────────────────────────────────
# CELDA 2 — Funciones del modelo
# Correr siempre al abrir el notebook
# ─────────────────────────────────────────────

def get_team_stats_v2(team, date, df, n_recent=5):
    home_all  = df[(df['HomeTeam'] == team) & (df['Date'] < date)]
    away_all  = df[(df['AwayTeam'] == team) & (df['Date'] < date)]
    all_games = pd.concat([home_all, away_all]).sort_values('Date')
    if len(all_games) < 3: return None
    current_season = df[df['Date'] < date]['season'].iloc[-1]
    season_games   = pd.concat([
        home_all[home_all['season'] == current_season],
        away_all[away_all['season'] == current_season]
    ]).sort_values('Date')
    recent = all_games.tail(n_recent)

    def calc_stats(games, team):
        if len(games) == 0: return None
        gf=gc=sot_f=sot_c=wins=draws=home_wins=home_games=0
        for _, r in games.iterrows():
            ih = r['HomeTeam'] == team
            gf    += r['FTHG'] if ih else r['FTAG']
            gc    += r['FTAG'] if ih else r['FTHG']
            sot_f += r['HST']  if ih else r['AST']
            sot_c += r['AST']  if ih else r['HST']
            won  = (ih and r['FTR']=='H') or (not ih and r['FTR']=='A')
            drew = r['FTR'] == 'D'
            if won:  wins  += 1
            if drew: draws += 1
            if ih:
                home_games += 1
                if r['FTR'] == 'H': home_wins += 1
        m = len(games)
        return {'gf_pg': gf/m, 'gc_pg': gc/m, 'dif_goles': (gf-gc)/m,
                'sot_f_pg': sot_f/m, 'sot_c_pg': sot_c/m,
                'win_rate': wins/m, 'draw_rate': draws/m,
                'home_wr': home_wins/home_games if home_games > 0 else wins/m}

    rs = calc_stats(recent, team)
    ss = calc_stats(season_games, team) if len(season_games) >= 3 else rs
    if rs is None: return None
    return {f'recent_{k}': rs[k] for k in rs} | {f'season_{k}': ss[k] if ss else rs[k] for k in rs}


def get_h2h_stats(home_team, away_team, date, df, n=10):
    h2h = df[
        ((df['HomeTeam']==home_team)&(df['AwayTeam']==away_team)) |
        ((df['HomeTeam']==away_team)&(df['AwayTeam']==home_team))
    ][lambda x: x['Date'] < date].tail(n)
    if len(h2h) < 3:
        return {'h2h_home_wr': 0.33, 'h2h_draw_rate': 0.33, 'h2h_away_wr': 0.33}
    hw = ((h2h['HomeTeam']==home_team)&(h2h['FTR']=='H')).sum() + \
         ((h2h['AwayTeam']==home_team)&(h2h['FTR']=='A')).sum()
    dr = (h2h['FTR']=='D').sum(); m = len(h2h)
    return {'h2h_home_wr': hw/m, 'h2h_draw_rate': dr/m, 'h2h_away_wr': (m-hw-dr)/m}


def get_tabla_posicion(team, date, df):
    season = df[df['Date'] < date]['season'].iloc[-1]
    ps     = df[(df['season']==season)&(df['Date']<date)]
    tabla  = []
    for eq in pd.concat([ps['HomeTeam'], ps['AwayTeam']]).unique():
        hp = ps[ps['HomeTeam']==eq]; ap = ps[ps['AwayTeam']==eq]
        tabla.append({'equipo': eq,
                      'pts': (hp['FTR']=='H').sum()*3+(hp['FTR']=='D').sum() +
                             (ap['FTR']=='A').sum()*3+(ap['FTR']=='D').sum(),
                      'gf': hp['FTHG'].sum()+ap['FTAG'].sum(),
                      'gc': hp['FTAG'].sum()+ap['FTHG'].sum(),
                      'pj': len(hp)+len(ap)})
    tdf = pd.DataFrame(tabla).sort_values('pts', ascending=False).reset_index(drop=True)
    tdf['pos'] = tdf.index + 1
    f = tdf[tdf['equipo']==team]
    if len(f)==0: return {'posicion': 9, 'pts_pg': 1.0, 'dif_goles_szn': 0, 'pct_posicion': 0.5}
    f = f.iloc[0]; pj = max(f['pj'],1)
    return {'posicion': f['pos'], 'pts_pg': f['pts']/pj,
            'dif_goles_szn': (f['gf']-f['gc'])/pj, 'pct_posicion': 1-(f['pos']/len(tdf))}


def predecir_partido_v3(home_team, away_team, fecha_str):
    """Predice H/D/A con probabilidades. Recomienda solo si confianza >55%."""
    fecha = pd.Timestamp(fecha_str)
    hs  = get_team_stats_v2(home_team, fecha, df)
    as_ = get_team_stats_v2(away_team, fecha, df)
    h2h = get_h2h_stats(home_team, away_team, fecha, df)
    ht2 = get_tabla_posicion(home_team, fecha, df)
    at2 = get_tabla_posicion(away_team, fecha, df)

    if hs is None or as_ is None:
        print(f'Sin historia suficiente: {home_team} o {away_team}')
        return None

    row = {}
    for k, v in hs.items():  row[f'h_{k}'] = v
    for k, v in as_.items(): row[f'a_{k}'] = v
    for k, v in h2h.items(): row[k] = v
    for k, v in ht2.items(): row[f'h_tabla_{k}'] = v
    for k, v in at2.items(): row[f'a_tabla_{k}'] = v
    row['dif_recent_wr']  = hs['recent_win_rate']  - as_['recent_win_rate']
    row['dif_season_wr']  = hs['season_win_rate']  - as_['season_win_rate']
    row['dif_recent_gf']  = hs['recent_gf_pg']     - as_['recent_gf_pg']
    row['dif_recent_gc']  = hs['recent_gc_pg']     - as_['recent_gc_pg']
    row['dif_recent_dif'] = hs['recent_dif_goles'] - as_['recent_dif_goles']
    row['dif_season_dif'] = hs['season_dif_goles'] - as_['season_dif_goles']
    row['home_advantage'] = hs['recent_home_wr']   - as_['recent_win_rate']
    row['dif_pts_pg']     = ht2['pts_pg']          - at2['pts_pg']
    row['dif_posicion']   = at2['posicion']        - ht2['posicion']

    proba     = model_v3.predict_proba(pd.DataFrame([row])[feature_cols_v3])[0]
    prob_A, prob_D, prob_H = proba[0], proba[1], proba[2]
    confianza = max(proba)
    pred      = le.inverse_transform([np.argmax(proba)])[0]
    usar      = confianza >= 0.55
    label     = {'H': f'{home_team} gana', 'D': 'Empate', 'A': f'{away_team} gana'}

    print(f"\n{'='*52}")
    print(f"  {home_team} vs {away_team}")
    print(f"{'='*52}")
    print(f"  Local gana     {home_team:<26} {prob_H:.1%}")
    print(f"  Empate                                      {prob_D:.1%}")
    print(f"  Visitante gana {away_team:<26} {prob_A:.1%}")
    print(f"{'─'*52}")
    print(f"  Prediccion:  {label[pred]}")
    print(f"  Confianza:   {confianza:.1%}  {'<- USAR' if usar else '<- IGNORAR'}")
    print(f"{'='*52}")

    return {'pred': pred, 'prob_H': prob_H, 'prob_D': prob_D,
            'prob_A': prob_A, 'confianza': confianza, 'usar': usar}


print('Funciones cargadas OK')

Funciones cargadas OK


In [5]:
# ─────────────────────────────────────────────
# CELDA 3 — Funciones de seguimiento
# Correr siempre al abrir el notebook
# ─────────────────────────────────────────────
predicciones = []

ARCHIVO_JSON = 'predicciones.json'
BASELINE     = 0.43  # Bundesliga: aprox 43% gana local


def registrar_prediccion(jornada, fecha, home, away, pred, prob_H, prob_D, prob_A, confianza):
    """Registra una prediccion ANTES del partido. Estado: pendiente."""
    predicciones.append({
        'jornada': jornada, 'fecha': fecha, 'home': home, 'away': away,
        'prediccion': pred,
        'prob_H': round(prob_H, 4), 'prob_D': round(prob_D, 4), 'prob_A': round(prob_A, 4),
        'confianza': round(confianza, 4),
        'resultado': None, 'correcto': None,
        'estado': 'pendiente'
    })
    print(f'Registrado: {home} vs {away} -> {pred} ({confianza:.1%})')


def registrar_resultado(home, away, resultado_real):
    """
    Registra el resultado DESPUES del partido y marca estado: verificado.
    resultado_real: 'H' (local), 'D' (empate), 'A' (visitante)
    """
    for p in predicciones:
        if p['home'] == home and p['away'] == away:
            p['resultado'] = resultado_real
            p['correcto']  = p['prediccion'] == resultado_real
            p['estado']    = 'verificado'
            print(f'{"ACERTADO" if p["correcto"] else "FALLADO"}: {home} vs {away} | '
                  f'Pred: {p["prediccion"]} | Real: {resultado_real}')
            return
    print(f'Partido no encontrado: {home} vs {away}')


def _calcular_resumen(lista):
    """Calcula resumen acumulado a partir de una lista de predicciones."""
    verificados = [p for p in lista if p['estado'] == 'verificado']
    total       = len(lista)
    n_verif     = len(verificados)
    acertados   = sum(1 for p in verificados if p['correcto'])
    accuracy    = round(acertados / n_verif, 4) if n_verif > 0 else 0.0
    return {
        'total_predicciones':      total,
        'partidos_verificados':    n_verif,
        'acertados':               acertados,
        'accuracy':                accuracy,
        'baseline':                BASELINE,
        'diferencia_vs_baseline':  round(accuracy - BASELINE, 4),
    }


def ver_seguimiento():
    """Muestra resumen acumulado + detalle por jornada."""
    if not predicciones:
        print('Sin predicciones registradas')
        return

    resumen  = _calcular_resumen(predicciones)
    accuracy = resumen['accuracy']

    print(f"\n{'='*52}")
    print(f"  RESUMEN ACUMULADO — Bundesliga")
    print(f"{'='*52}")
    print(f"  Total predicciones:    {resumen['total_predicciones']}")
    print(f"  Partidos verificados:  {resumen['partidos_verificados']}")
    print(f"  Acertados:             {resumen['acertados']}")
    print(f"  Accuracy real:         {accuracy:.1%}")
    print(f"  Baseline (siempre H):  {BASELINE:.1%}")
    print(f"  Diferencia:            {resumen['diferencia_vs_baseline']:+.1%}")

    # Detalle por jornada
    jornadas = sorted(set(p['jornada'] for p in predicciones))
    for j in jornadas:
        pj       = [p for p in predicciones if p['jornada'] == j]
        verif_j  = [p for p in pj if p['estado'] == 'verificado']
        acert_j  = sum(1 for p in verif_j if p['correcto'])
        acc_j    = acert_j / len(verif_j) if verif_j else None
        pendient = sum(1 for p in pj if p['estado'] == 'pendiente')

        print(f"\n{'─'*52}")
        acc_str = f'{acc_j:.1%}' if acc_j is not None else 'pendiente'
        print(f"  JORNADA {j}  |  {acert_j}/{len(verif_j)} verificados  |  accuracy: {acc_str}")
        if pendient:
            print(f"  ({pendient} pendiente{'s' if pendient > 1 else ''} de resultado)")
        print(f"{'─'*52}")
        for p in pj:
            if p['estado'] == 'verificado':
                icon = 'OK' if p['correcto'] else 'XX'
                print(f"  [{icon}] {p['home']} vs {p['away']}")
                print(f"       Pred: {p['prediccion']} ({p['confianza']:.1%}) | Real: {p['resultado']}")
            else:
                print(f"  [??] {p['home']} vs {p['away']}")
                print(f"       Pred: {p['prediccion']} ({p['confianza']:.1%}) | Pendiente")

    # Accuracy por nivel de confianza (solo verificados)
    verif_all = [p for p in predicciones if p['estado'] == 'verificado']
    if verif_all:
        import pandas as pd
        df_v = pd.DataFrame(verif_all)
        df_v['cb'] = pd.cut(df_v['confianza'], bins=[0.50, 0.55, 0.60, 0.65, 1.0],
                            labels=['50-55%', '55-60%', '60-65%', '>65%'])
        print(f"\n  Accuracy por nivel de confianza:")
        print(df_v.groupby('cb', observed=True).agg(
            partidos=('correcto', 'count'), accuracy=('correcto', 'mean')
        ).round(3).to_string())


def guardar_predicciones(archivo=ARCHIVO_JSON):
    """Guarda en disco con estructura: resumen_acumulado + jornadas."""
    # Cargar historial existente para no perder jornadas anteriores
    try:
        with open(archivo, 'r') as f:
            existente = json.load(f)
        jornadas_existentes = existente.get('jornadas', {})
    except FileNotFoundError:
        jornadas_existentes = {}

    # Agrupar predicciones actuales por jornada
    jornadas_nuevas = {}
    for p in predicciones:
        k = str(p['jornada'])
        if k not in jornadas_nuevas:
            jornadas_nuevas[k] = []
        entry = {**p, 'correcto': bool(p['correcto']) if p['correcto'] is not None else None}
        jornadas_nuevas[k].append(entry)

    # Merge: las jornadas nuevas sobreescriben las existentes
    jornadas_merge = {**jornadas_existentes}
    for k, preds in jornadas_nuevas.items():
        verif = [p for p in preds if p['estado'] == 'verificado']
        acert = sum(1 for p in verif if p['correcto'])
        jornadas_merge[k] = {
            'partidos_usados':   len(preds),
            'acertados':         acert,
            'accuracy_jornada':  round(acert / len(verif), 4) if verif else None,
            'predicciones':      preds,
        }

    # Recalcular resumen acumulado sobre TODAS las jornadas
    todas = [p for j in jornadas_merge.values() for p in j['predicciones']]
    resumen = _calcular_resumen(todas)

    datos = {
        'liga':               'Bundesliga',
        'resumen_acumulado':  resumen,
        'jornadas':           jornadas_merge,
    }
    with open(archivo, 'w') as f:
        json.dump(datos, f, indent=2, ensure_ascii=False)
    print(f'Guardado: {len(predicciones)} predicciones en {archivo}')
    print(f'  Jornadas en archivo: {sorted(jornadas_merge.keys())}')


def cargar_predicciones(archivo=ARCHIVO_JSON):
    """Carga desde disco todas las predicciones de todas las jornadas."""
    global predicciones
    try:
        with open(archivo, 'r') as f:
            datos = json.load(f)
        # Aplanar todas las jornadas en una sola lista
        predicciones = [
            p
            for j in datos.get('jornadas', {}).values()
            for p in j.get('predicciones', [])
        ]
        print(f'Cargado: {len(predicciones)} predicciones '
              f'({len(datos["jornadas"])} jornadas)')
    except FileNotFoundError:
        print('No hay predicciones guardadas — lista vacia')


print('Funciones de seguimiento cargadas OK')


Funciones de seguimiento cargadas OK


In [6]:
# ─────────────────────────────────────────────
# CELDA 4 — Predecir jornada
# Editar: partidos_jornada y fecha_jornada
#
# Nombres exactos como aparecen en el dataset
# (ver lista de equipos en Celda 1)
# ─────────────────────────────────────────────

# ── EDITAR AQUI ──────────────────────────────
# ── JORNADA 29
# partidos_jornada = [
#     ('Augsburg',      'Hoffenheim'),     # Viernes  10 abr
#     ('Dortmund',      'Leverkusen'),     # Sábado   11 abr
#     ('RB Leipzig',    "M'gladbach"),     # Sábado   11 abr
#     ('Wolfsburg',     'Ein Frankfurt'),  # Sábado   11 abr
#     ('Heidenheim',    'Union Berlin'),   # Sábado   11 abr
#     ('St Pauli',      'Bayern Munich'),  # Sábado   11 abr
#     ('FC Koln',       'Werder Bremen'),  # Domingo  12 abr
#     ('Stuttgart',     'Hamburg'),        # Domingo  12 abr
#     ('Mainz',         'Freiburg'),       # Domingo  12 abr
# ]
# fecha_jornada = '2026-04-10'
# ─────────────────────────────────────────────

partidos_jornada = [
    ('St Pauli',      'FC Koln'),         # Viernes  17 abr
    ('Leverkusen',    'Augsburg'),         # Sábado   18 abr
    ('Werder Bremen', 'Hamburg'),          # Sábado   18 abr
    ('Union Berlin',  'Wolfsburg'),        # Sábado   18 abr
    ('Hoffenheim',    'Dortmund'),         # Sábado   18 abr
    ('Ein Frankfurt', 'RB Leipzig'),       # Sábado   18 abr
    ('Freiburg',      'Heidenheim'),       # Domingo  19 abr
    ('Bayern Munich', 'Stuttgart'),        # Domingo  19 abr
    ("M'gladbach",    'Mainz'),            # Domingo  19 abr
]
fecha_jornada = '2026-04-17'

resultados = {}
for home, away in partidos_jornada:
    res = predecir_partido_v3(home, away, fecha_jornada)
    if res:
        resultados[f'{home} vs {away}'] = res


  St Pauli vs FC Koln
  Local gana     St Pauli                   31.8%
  Empate                                      24.5%
  Visitante gana FC Koln                    43.8%
────────────────────────────────────────────────────
  Prediccion:  FC Koln gana
  Confianza:   43.8%  <- IGNORAR

  Leverkusen vs Augsburg
  Local gana     Leverkusen                 56.7%
  Empate                                      24.0%
  Visitante gana Augsburg                   19.3%
────────────────────────────────────────────────────
  Prediccion:  Leverkusen gana
  Confianza:   56.7%  <- USAR

  Werder Bremen vs Hamburg
  Local gana     Werder Bremen              44.1%
  Empate                                      30.4%
  Visitante gana Hamburg                    25.5%
────────────────────────────────────────────────────
  Prediccion:  Werder Bremen gana
  Confianza:   44.1%  <- IGNORAR

  Union Berlin vs Wolfsburg
  Local gana     Union Berlin               50.6%
  Empate                                

In [12]:
# ─────────────────────────────────────────────
# CELDA 5 — Registrar predicciones a usar
# Solo registrar las marcadas con <- USAR
# ─────────────────────────────────────────────
cargar_predicciones()

# ── EDITAR AQUI — solo confianza >55% ────────
# Ejemplo — reemplazar con los resultados reales de la Celda 4:
#
# registrar_prediccion(
#     jornada=29, fecha='2026-04-11',
#     home='RB Leipzig', away="M'gladbach",
#     pred='H', prob_H=0.595, prob_D=0.292, prob_A=0.113,
#     confianza=0.595
# )

# registrar_prediccion(
#     jornada=29, fecha='2026-04-11',
#     home='St Pauli', away='Bayern Munich',
#     pred='A', prob_H=0.128, prob_D=0.152, prob_A=0.720,
#     confianza=0.720
# )

# registrar_prediccion(
#     jornada=29, fecha='2026-04-12',
#     home='Stuttgart', away='Hamburg',
#     pred='H', prob_H=0.661, prob_D=0.269, prob_A=0.070,
#     confianza=0.661
# )
# ─────────────────────────────────────────────

registrar_prediccion(
    jornada=30, fecha='2026-04-18',
    home='Leverkusen', away='Augsburg',
    pred='H', prob_H=0.567, prob_D=0.240, prob_A=0.193,
    confianza=0.567
)

guardar_predicciones()
print()
ver_seguimiento()

Cargado: 3 predicciones (1 jornadas)
Registrado: Leverkusen vs Augsburg -> H (56.7%)
Guardado: 4 predicciones en predicciones.json
  Jornadas en archivo: ['29', '30']


  RESUMEN ACUMULADO — Bundesliga
  Total predicciones:    4
  Partidos verificados:  3
  Acertados:             3
  Accuracy real:         100.0%
  Baseline (siempre H):  43.0%
  Diferencia:            +57.0%

────────────────────────────────────────────────────
  JORNADA 29  |  3/3 verificados  |  accuracy: 100.0%
────────────────────────────────────────────────────
  [OK] RB Leipzig vs M'gladbach
       Pred: H (59.5%) | Real: H
  [OK] St Pauli vs Bayern Munich
       Pred: A (72.0%) | Real: A
  [OK] Stuttgart vs Hamburg
       Pred: H (66.1%) | Real: H

────────────────────────────────────────────────────
  JORNADA 30  |  0/0 verificados  |  accuracy: pendiente
  (1 pendiente de resultado)
────────────────────────────────────────────────────
  [??] Leverkusen vs Augsburg
       Pred: H (56.7%) | Pendiente

  Accuracy

In [8]:
# ─────────────────────────────────────────────
# CELDA 6 — Registrar resultados reales
# Correr DESPUES de cada partido
#
# H = gana local    D = empate    A = gana visitante
# ─────────────────────────────────────────────
# cargar_predicciones()

# # ── EDITAR AQUI — cambiar por resultado real ─
# registrar_resultado('Bayern Munich', 'Dortmund', 'H')  # <- cambiar despues del partido
# registrar_resultado('Leverkusen',    'RB Leipzig', 'H')  # <- cambiar despues del partido
# # ─────────────────────────────────────────────

# guardar_predicciones()

In [9]:
# ─────────────────────────────────────────────
# CELDA 7 — Ver resumen completo
# Correr en cualquier momento
# ─────────────────────────────────────────────
cargar_predicciones()
ver_seguimiento()

Cargado: 6 predicciones (1 jornadas)

  RESUMEN ACUMULADO — Bundesliga
  Total predicciones:    6
  Partidos verificados:  3
  Acertados:             3
  Accuracy real:         100.0%
  Baseline (siempre H):  43.0%
  Diferencia:            +57.0%

────────────────────────────────────────────────────
  JORNADA 29  |  3/3 verificados  |  accuracy: 100.0%
  (3 pendientes de resultado)
────────────────────────────────────────────────────
  [OK] RB Leipzig vs M'gladbach
       Pred: H (59.5%) | Real: H
  [OK] St Pauli vs Bayern Munich
       Pred: A (72.0%) | Real: A
  [OK] Stuttgart vs Hamburg
       Pred: H (66.1%) | Real: H
  [??] RB Leipzig vs M'gladbach
       Pred: H (59.5%) | Pendiente
  [??] St Pauli vs Bayern Munich
       Pred: A (72.0%) | Pendiente
  [??] Stuttgart vs Hamburg
       Pred: H (66.1%) | Pendiente

  Accuracy por nivel de confianza:
        partidos  accuracy
cb                        
55-60%         1       1.0
>65%           2       1.0
